<a href="https://colab.research.google.com/github/c-rhoden/analysis-of-loss-of-p53-on-gene-function/blob/main/Copy_of_01_cohort_table.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧹 Notebook 01 — Clean the cohort, describe the patients

**Produces:** a clean patient table, and your "Table 1".

---

> **This notebook stands alone.** Nothing in Notebook 02 uses its
> output. If you get stuck here, go to `02_p53_analysis` — it has its
> own, already-clean data file.

## Why you are doing this

Open any clinical paper and the first table is always the same: *who
were the patients?* How many, how long were they followed, how many
died. It is called **Table 1**, and no reviewer reads past it if it is
wrong.

You have the raw patient export exactly as it comes out of the database
— barcodes instead of patient IDs, numeric codes instead of sample
labels, follow-up time in days, and missing values written four
different ways. **Turn that into something you could put Table 1 on.**

> ### ⚠️ Run every cell, in order, from the top
>
> Your functions only exist once you have **run** the cell that defines
> them. If you edit a function, you must re-run its cell or Python
> carries on using the old version — which is the single most confusing
> bug in notebook work.
>
> If anything stops making sense, use **Runtime → Restart and run all**
> (Colab) or **Kernel → Restart & Run All** (VS Code) and start clean.

## Run this first

> ✏️ **Before you run the cell below, add your Sussex username.**
>
> Scroll to the **bottom** of the cell — **line 66 of 89** —
> and type your Sussex username (the start of your Sussex email, the
> one you log in with) between the quotes:
>
> ```python
> STUDENT_ID = "ab123"   # <-- your Sussex username
> ```
>
> Then run the cell (**Shift + Enter**). It prints your gene and cancer
> type. Change nothing else in that cell.
>
> 💡 To see line numbers in Colab: **Tools → Settings → Editor → Show
> line numbers**.

In [1]:
# Run this first. It downloads your data and sets up the self-checks.
import hashlib
import json
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

DATA_URL = "https://raw.githubusercontent.com/HocheggerLab/tcga-capstone-2026/main/data/"

def fetch(filename):
    """Return a local data path, downloading it in Colab if needed."""
    repo_path = Path("data") / filename
    working_path = Path(filename)

    if repo_path.exists():                   # running from the repo
        return repo_path
    if not working_path.exists():            # running in Colab
        urllib.request.urlretrieve(DATA_URL + filename, working_path)
    return working_path


def my_variant(student_id):
    """Look up your gene and cancer type from your Sussex username."""
    number = str(student_id).strip().lower()
    if not number:
        raise ValueError("Type your Sussex username into "
                         "STUDENT_ID, then run this cell again.")
    key = hashlib.pbkdf2_hmac("sha256", number.encode(), b'tcga-capstone-2026',
                              2_000_000).hex()[:16]
    lookup = json.loads(Path(fetch("variant_lookup.json")).read_text())
    if key not in lookup:
        raise ValueError(f"{number} is not on the class list. Check it "
                         "against Canvas; if it is right, email the convenor.")
    gene, cancer_type = lookup[key]
    print(f"{number}: your gene is {gene}, your cancer type is {cancer_type}")
    return gene, cancer_type


def run_checks(checks):
    """Run a dict of {description: test} and report what passed.

    Each test is a zero-argument function returning True or False. A
    test that raises is reported rather than stopping the notebook, so
    you always see the full picture.
    """
    passed = 0
    for description, test in checks.items():
        try:
            ok = bool(test())
        except Exception as exc:
            print(f"\u274c {description}")
            print(f"     raised {type(exc).__name__}: {exc}")
            continue
        print(f"{'\u2705' if ok else '\u274c'} {description}")
        passed += ok
    print()
    print(f"{passed} of {len(checks)} checks passed")
    if passed < len(checks):
        print("Keep going - fix the crosses and run this cell again.")


STUDENT_ID = "cr505"   # <-- your Sussex username, e.g. "ab123"

GENE, CANCER_TYPE = my_variant(STUDENT_ID)

MY_PHENOTYPE = fetch(f"phenotype_raw_{CANCER_TYPE}.csv")
EXAMPLE_PHENOTYPE = fetch("phenotype_raw_example.csv")

# Given to you - the codes used in the raw export.
SAMPLE_TYPES = {
    "01": "Primary Tumor",
    "02": "Recurrent Tumor",
    "03": "Primary Blood Derived Cancer - Peripheral Blood",
    "05": "Additional - New Primary",
    "06": "Metastatic",
    "07": "Additional Metastatic",
    "11": "Solid Tissue Normal",
}

# Given to you - the four ways this export writes "missing".
MISSING = ["not reported", "--", ""]

raw = pd.read_csv(MY_PHENOTYPE)
print(raw.shape)
raw.head()

cr505: your gene is TP53I3, your cancer type is LUAD
(589, 5)


,sampleID,primary disease or tissue,sample_type_code,days_to_last_followup,vital_status
0,TCGA-55-7903-01,lung adenocarcinoma,1,567.0,Alive
1,TCGA-73-7499-01,Lung Adenocarcinoma,1,not reported,Dead
2,TCGA-62-8402-01,lung adenocarcinoma,1,1498.0,Dead
3,TCGA-86-8055-01,lung adenocarcinoma,1,124.0,Dead
4,TCGA-62-A46O-01,Lung Adenocarcinoma,1,1454.0,Dead


## Look at the mess before you touch it

Never start cleaning before you have looked.

In [2]:
print((raw["sample_type_code"][570:]))

570    11
571     1
572     1
573     1
574     1
575    11
576     1
577     1
578     1
579     1
580    11
581     1
582    11
583     1
584     1
585     1
586     1
587    11
588     1
Name: sample_type_code, dtype: int64


In [3]:
# DEMO - what is actually in each column?
for col in raw.columns:
    print(f"--- {col}")
    print(raw[col].astype(str).str.strip().value_counts().head(4).to_string())
    print()

--- sampleID
sampleID
TCGA-78-7150-01    2
TCGA-05-4434-01    2
TCGA-44-A47G-01    2
TCGA-86-A456-01    2

--- primary disease or tissue
primary disease or tissue
LUNG ADENOCARCINOMA    198
Lung Adenocarcinoma    196
lung adenocarcinoma    195

--- sample_type_code
sample_type_code
1     525
11     62
2       2

--- days_to_last_followup
days_to_last_followup
nan             27
not reported    19
--              18
791.0            4

--- vital_status
vital_status
Alive    372
Dead     217



### What you should have spotted

1. **`sampleID` is a TCGA barcode**, like `TCGA-05-4244-01`. The first
   **12 characters** are the patient — `TCGA-05-4244`. One patient can
   appear more than once.
2. **`primary disease or tissue` is inconsistent** — upper case, lower
   case, stray spaces. Three spellings of one thing is three categories
   to pandas.
3. **`sample_type_code` is numeric.** `01` is a primary tumour, `11` is
   normal tissue, `06` a metastasis. `SAMPLE_TYPES` translates them.
4. **`days_to_last_followup` is not a number.** Some rows say
   `not reported`, some `--`, some are blank. Leave them and the column
   stays text, and you cannot take a median of text.
5. **There are duplicate rows.** The same `sampleID` appears twice.

## Task 1a — `load_and_clean`

**Takes** `path` — the path to a raw phenotype CSV.

**Returns** a DataFrame with **exactly these columns, in this order**:

```
sample_id, patient_id, cancer_type, sample_type, is_tumour, followup_years, vital_status
```

| column | what it must contain |
|---|---|
| `sample_id` | the barcode, unchanged |
| `patient_id` | first 12 characters of the barcode |
| `cancer_type` | stripped of spaces, upper case |
| `sample_type` | the readable label, via `SAMPLE_TYPES` |
| `is_tumour` | `True` / `False`, a real boolean — everything except `Solid Tissue Normal` |
| `followup_years` | days ÷ 365.25, rounded to 1 dp, `NaN` where missing |
| `vital_status` | `Alive`, `Dead`, or `NaN` |

Drop rows with a duplicated `sample_id`, keeping the first, and reset
the index.

> 💡 Do not start by filling in the function. Add a cell, load the file,
> and work out the seven columns one at a time in raw code. Wrap it up
> only once it works — see *How to build each function* in
> `00_start_here`.

**Three things that catch people out:** the sentinels must become `NaN`
*before* you convert to a number; `followup_years` must end up a float,
not text; and the column order is part of the contract.

In [20]:
#print(raw.head())
# step 1 fix the text in the primary disease or tissue column
#print(raw["primary disease or tissue"][:5])
def clean_sample_name(condition:str)-> str:
  """Cleaning the condition column so they are all upper case with the same spacing"""
  for condition in (raw["primary disease or tissue"]):
    raw["primary disease or tissue"] = "Lung Adenocarcinoma"
  return raw["primary disease or tissue"]
print(clean_sample_name(raw[:5]))

def duplicates(sample_id:str) -> str: # manipulating data with pandas course on datacamp
  """Removing samples that show up more than once in the dataset"""
  for sample in raw["sampleID"]:
    samples = raw.drop_duplicates(subset="sampleID")
  return samples
print(duplicates(raw["sampleID"]))

def years_interger(days_to_last_followup):
  """coverting the days to last followup column to intergers and into years"""
  for days in raw["days_to_last_followup"]:
   raw["days_to_last_followup"].replace("not reported", np.nan)
   raw["days_to_last_followup"].replace("--", np.nan)
  return
raw["days_to_last_followup"].value_counts()

0      Lung Adenocarcinoma
1      Lung Adenocarcinoma
2      Lung Adenocarcinoma
3      Lung Adenocarcinoma
4      Lung Adenocarcinoma
              ...         
584    Lung Adenocarcinoma
585    Lung Adenocarcinoma
586    Lung Adenocarcinoma
587    Lung Adenocarcinoma
588    Lung Adenocarcinoma
Name: primary disease or tissue, Length: 589, dtype: object
            sampleID primary disease or tissue  sample_type_code  \
0    TCGA-55-7903-01       Lung Adenocarcinoma                 1   
1    TCGA-73-7499-01       Lung Adenocarcinoma                 1   
2    TCGA-62-8402-01       Lung Adenocarcinoma                 1   
3    TCGA-86-8055-01       Lung Adenocarcinoma                 1   
4    TCGA-62-A46O-01       Lung Adenocarcinoma                 1   
..               ...                       ...               ...   
584  TCGA-50-8460-01       Lung Adenocarcinoma                 1   
585  TCGA-55-A4DF-01       Lung Adenocarcinoma                 1   
586  TCGA-91-6835-01       Lung

,count
days_to_last_followup,
791.0,4
385.0,4
0.0,4
761.0,4
653.0,3
...,...
567.0,1
2318.0,1
243.0,1


In [6]:
# using .value_counts() to check repeat samples were actually removed
print(raw["sampleID"].value_counts())
print(len(raw["sampleID"])) # 589
cleaned = duplicates(raw["sampleID"])
print(cleaned.value_counts())
print(len(cleaned)) # 574
# there was 15 repeated samples in raw that were removed

sampleID
TCGA-78-7150-01    2
TCGA-05-4434-01    2
TCGA-44-A47G-01    2
TCGA-86-A456-01    2
TCGA-97-7547-01    2
                  ..
TCGA-67-6217-01    1
TCGA-49-4494-01    1
TCGA-MN-A4N1-01    1
TCGA-93-7348-01    1
TCGA-95-7567-01    1
Name: count, Length: 574, dtype: int64
589
sampleID         primary disease or tissue  sample_type_code  days_to_last_followup  vital_status
TCGA-05-4244-01  Lung Adenocarcinoma        1                 0.0                    Alive           1
TCGA-05-4250-01  Lung Adenocarcinoma        1                 121.0                  Dead            1
TCGA-05-4382-01  Lung Adenocarcinoma        1                 607.0                  Alive           1
TCGA-05-4384-01  Lung Adenocarcinoma        1                 426.0                  Alive           1
TCGA-05-4389-01  Lung Adenocarcinoma        1                 1369.0                 Alive           1
                                                                                                    ..
T

In [7]:
def load_and_clean(path):
    # TODO - your turn
    raise NotImplementedError("load_and_clean")

In [8]:
# CHECK - runs your function on the worked example (CDKN1A in BRCA)
EXPECTED_COLUMNS = ["sample_id", "patient_id", "cancer_type",
                    "sample_type", "is_tumour", "followup_years",
                    "vital_status"]

checked = load_and_clean(EXAMPLE_PHENOTYPE)

run_checks({
    "returns a DataFrame":
        lambda: isinstance(checked, pd.DataFrame),
    "columns are correct, and in the right order":
        lambda: list(checked.columns) == EXPECTED_COLUMNS,
    "patient_id is the first 12 characters of the barcode":
        lambda: (checked["sample_id"].str[:12] == checked["patient_id"]).all(),
    "duplicate sample_id rows were dropped":
        lambda: checked["sample_id"].is_unique,
    "no sentinel strings survived":
        lambda: not any((checked.astype(str) == s).any().any()
                        for s in MISSING),
    "followup_years is a float, not text":
        lambda: checked["followup_years"].dtype == float,
    "is_tumour is a real boolean":
        lambda: checked["is_tumour"].dtype == bool,
    "row count matches the worked example (1212)":
        lambda: len(checked) == 1212,
    "tumour/normal split matches (1099/113)":
        lambda: (int(checked["is_tumour"].sum()) == 1099
                 and int((~checked["is_tumour"]).sum()) == 113),
})

NotImplementedError: load_and_clean

Once that passes, run it on your own data:

In [ ]:
clean = load_and_clean(MY_PHENOTYPE)
clean.head()

## Task 1b — `describe_cohort`

Now the actual deliverable.

**Takes** `clean` — a table in the shape `load_and_clean` returns.

**Returns** a dict of the six numbers that make up Table 1:

```python
{"n_patients": int, "n_samples": int, "n_tumour": int,
 "n_normal": int, "median_followup_years": float, "n_deceased": int}
```

`n_patients` and `n_samples` are **not** the same number, and noticing
why is half the point of this task.

In [ ]:
def describe_cohort(clean):
    # TODO - your turn
    raise NotImplementedError("describe_cohort")

In [ ]:
# CHECK - against the worked example
EXPECTED_COHORT = {
    "n_patients": 1093,
    "n_samples": 1212,
    "n_tumour": 1099,
    "n_normal": 113,
    "median_followup_years": 2.5,
    "n_deceased": 198,
}

summary = describe_cohort(checked)

run_checks({
    "returns a dict":
        lambda: isinstance(summary, dict),
    "has exactly the six required keys":
        lambda: set(summary) == set(EXPECTED_COHORT),
    "samples split into tumour and normal with none left over":
        lambda: summary["n_tumour"] + summary["n_normal"] == summary["n_samples"],
    "there are no more patients than samples":
        lambda: summary["n_patients"] <= summary["n_samples"],
    "every value matches the worked example":
        lambda: all(summary[k] == v for k, v in EXPECTED_COHORT.items()),
})

In [ ]:
my_cohort = describe_cohort(clean)
my_cohort

## Write your Table 1

Use your own numbers. This is marked on whether it is **correct and
readable**, not on how you computed it.

### Table 1 — write yours here

| | |
|---|---|
| Patients | |
| Samples | |
| Primary tumour / normal | |
| Median follow-up (years) | |
| Deceased | |

*One or two sentences describing the cohort — and say why the patient
count and the sample count differ.*